# FastVision: 3-line visual token pruning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Cekaru/fastvision/blob/main/examples/llava_colab.ipynb)

Wrap LLaVA-1.5 so redundant visual tokens are pruned before the language model — **no fine-tuning, no architecture edits** — then compare latency, memory and output quality. Runs on the free T4 GPU.

In [ ]:
!pip install -q git+https://github.com/Cekaru/fastvision.git accelerate

In [ ]:
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

MODEL_ID = "llava-hf/llava-1.5-7b-hf"
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.float16, device_map="cuda"
)
processor = AutoProcessor.from_pretrained(MODEL_ID)

In [ ]:
import requests
from PIL import Image

image = Image.open(requests.get(
    "http://images.cocodataset.org/val2017/000000039769.jpg", stream=True
).raw)

messages = [{"role": "user", "content": [
    {"type": "image", "image": image},
    {"type": "text", "text": "What is happening in this image?"},
]}]
inputs = processor.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=True,
    return_dict=True, return_tensors="pt",
).to(model.device)
image

## Baseline (576 visual tokens)

In [ ]:
import time

def timed_generate(tag):
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize(); t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=60, do_sample=False)
    torch.cuda.synchronize()
    text = processor.batch_decode(out, skip_special_tokens=True)[0]
    print(f"[{tag}] {time.perf_counter()-t0:.2f}s, "
          f"peak {torch.cuda.max_memory_allocated()/2**20:.0f} MB")
    print(text.split('ASSISTANT:')[-1].strip() if 'ASSISTANT:' in text else text)

timed_generate("baseline")

## The 3 lines

In [ ]:
from fastvision import FastVisionWrapper

model = FastVisionWrapper(model, keep_ratio=0.1)   # 576 -> ~58 tokens
timed_generate("keep_ratio=0.1")
print(model.fastvision.stats)

## Try strategies and budgets

The config is live on `model.fastvision` — no re-wrapping needed for budget changes.

In [ ]:
model.fastvision.keep_ratio = 0.05
timed_generate("keep_ratio=0.05")

model.unwrap()
model = FastVisionWrapper(model, keep_ratio=0.1, strategy="tome")  # merge, don't drop
timed_generate("tome @ 0.1")
model.unwrap()